In [1]:
# Install requirements quietly
!pip install -q yfinance plotly scipy pandas numpy

import yfinance as yf
import pandas as pd
import numpy as np
import scipy.optimize as sco
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Suppress annoying warnings
warnings.filterwarnings('ignore')

print("Fetching financial data... (this takes about 5 seconds)")

# 1. Fetch Data Safely (bypassing the new Yahoo Finance bug)
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
data = yf.download(tickers, start='2020-01-01', end='2024-01-01', progress=False)

# Extract just the 'Close' prices safely
if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close']
else:
    prices = data

prices = prices.ffill().dropna()
returns = prices.pct_change().dropna()

print("Data fetched successfully! Calculating optimized portfolio...")

# 2. Portfolio Optimization Engine
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

def negative_sharpe(weights):
    p_ret = np.sum(mean_returns * weights)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - 0.04) / p_std # 0.04 is the risk-free rate

num_assets = len(tickers)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0.0, 1.0) for _ in range(num_assets))
initial_guess = num_assets * [1./num_assets]

opt_results = sco.minimize(negative_sharpe, initial_guess, method='SLSQP', bounds=bounds, constraints=constraints)
optimal_weights = opt_results.x

# Print Text Results
print("\n" + "="*40)
print("🏆 OPTIMIZED PORTFOLIO (MAX SHARPE)")
print("="*40)
for ticker, weight in zip(tickers, optimal_weights):
    print(f"{ticker}: {weight*100:.2f}%")
print("="*40 + "\n")

# 3. Backtesting Engine (Testing on AAPL)
aapl = prices[['AAPL']].copy()
aapl.columns = ['Price']

# Calculate 50-day and 200-day Moving Averages
aapl['SMA50'] = aapl['Price'].rolling(50).mean()
aapl['SMA200'] = aapl['Price'].rolling(200).mean()

# Trading Signal: 1 if SMA50 is above SMA200, else 0
aapl['Signal'] = np.where(aapl['SMA50'] > aapl['SMA200'], 1.0, 0.0)

# Calculate Returns
aapl['Market_Returns'] = aapl['Price'].pct_change()
aapl['Strategy_Returns'] = aapl['Market_Returns'] * aapl['Signal'].shift(1)
aapl = aapl.dropna()

# Cumulative Returns for the Chart
aapl['Cumulative_Market'] = (1 + aapl['Market_Returns']).cumprod()
aapl['Cumulative_Strategy'] = (1 + aapl['Strategy_Returns']).cumprod()

# 4. Generate Interactive Dashboard
fig = make_subplots(rows=2, cols=1, subplot_titles=("AAPL: SMA Crossover Strategy", "Strategy vs Market Cumulative Returns"))

# Top Chart: Price and Moving Averages
fig.add_trace(go.Scatter(x=aapl.index, y=aapl['Price'], name='AAPL Price', line=dict(color='black', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=aapl.index, y=aapl['SMA50'], name='50-Day SMA', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=aapl.index, y=aapl['SMA200'], name='200-Day SMA', line=dict(color='red', width=1)), row=1, col=1)

# Bottom Chart: Performance Comparison
fig.add_trace(go.Scatter(x=aapl.index, y=aapl['Cumulative_Market'], name='Buy & Hold Market', line=dict(color='gray')), row=2, col=1)
fig.add_trace(go.Scatter(x=aapl.index, y=aapl['Cumulative_Strategy'], name='Our Trading Strategy', line=dict(color='green', width=2.5)), row=2, col=1)

fig.update_layout(height=800, title_text="QuantTerminal: Live Dashboard", template="plotly_white")
fig.show()

Fetching financial data... (this takes about 5 seconds)
Data fetched successfully! Calculating optimized portfolio...

🏆 OPTIMIZED PORTFOLIO (MAX SHARPE)
AAPL: 25.04%
MSFT: 0.00%
GOOGL: 0.00%
AMZN: 18.31%
TSLA: 56.65%

